## HPC Model Quality Review

This section evaluates all model runs in `models/hpc_runs` and compares them consistently.

Flow:
1. Build one shared evaluation data plan (top cell below).
2. Run one evaluation cell per model folder (or reuse existing results if a model type is not supported by `load_checkpoint`).
3. Compare all available `eval_results.json` files with `compare_eval`.

### Helpers

In [56]:
from pathlib import Path
from IPython.display import Markdown, display, IFrame
from evaluation.load_checkpoint import run_checkpoint_evaluation
from evaluation.compare_eval import ModelComparator
import random
import os
import matplotlib.pyplot as plt
import numpy as np
import json
import yaml


try:
    from tabulate import tabulate as _tabulate
except Exception:
    _tabulate = None

PROJECT_ROOT = Path('/Users/clarareginekold/Desktop/dtu/THESIS/rasmus-clara-synthetic-data').resolve()
HPC_RUNS_DIR = PROJECT_ROOT / 'models' / 'hpc_runs'
CONFIG_PATH = PROJECT_ROOT / 'config' / 'config.yaml'
BASELINE_CACHE_DIR = PROJECT_ROOT / 'evaluation_comparison_hpc_runs'
BASELINE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Force project root cwd so relative conditioner paths (holiday/heatpump) resolve correctly.
os.chdir(PROJECT_ROOT)

# Fixed run list so this notebook keeps one dedicated cell per model.
MODEL_RUNS = [
    'AutoRegressiveTransformerVAE_AutoregressiveTransformerCoder_20260415_163139',
    'CVAE_DictionaryCoder_20260416_082400',
    'CVAE_DictionaryCoder_20260416_085408_hierarchy',
    'CVAE_DictionaryCoder_20260416_230553_vamp',
    'CVAE_GaussianCoder_20260416_085358',
    'DiffusionModel_ConditionalUNet1D_20260415_172958',
    'SetGuideVAE_DictionaryCoder_20260416_103152',
]

DATA_SOURCES = {
    '1phase_hourly': PROJECT_ROOT / 'data' / 'parq_hourly',
    '3phase_15min': PROJECT_ROOT / 'data' / 'raw_profiles',
}

# Canonical metric naming so model/baseline files align even when keys differ.
_CANONICAL_METRIC_BY_KEY = {
    'reconstructionrmse': 'Reconstruction RMSE',
    'reconstructionmse': 'Reconstruction MSE',
    'reconstructionloglikelihood': 'Reconstruction Log-Likelihood',
    'reconstructionlog-likelihood': 'Reconstruction Log-Likelihood',
    'flexibilitydistance': 'Flexibility Distance',
    'maximummeandiscrepancy(mmd)': 'Maximum Mean Discrepancy (MMD)',
    'maximummeandiscrepancy': 'Maximum Mean Discrepancy (MMD)',
    'acfdistributionmatch(ks)': 'ACF Distribution Match (KS)',
    'classicalstatistics': 'Classical Statistics',
    'tstr(trainonsynthetic,testonreal)': 'TSTR (Train on Synthetic, Test on Real)',
    'tstr': 'TSTR (Train on Synthetic, Test on Real)',
    'aggregatedprofiles(flexibilitydistance)': 'Aggregated Profiles (Flexibility Distance)',
    'aggregatedprofiles': 'Aggregated Profiles (Flexibility Distance)',
    'per-conditionevaluation': 'Per-Condition Evaluation',
    'perconditionevaluation': 'Per-Condition Evaluation',
    'perconditioneval': 'Per-Condition Evaluation',
    'visualevaluations': 'Visual Evaluations',
}

def _norm_metric_key(name: str) -> str:
    return ''.join(ch for ch in str(name).lower() if ch.isalnum() or ch in {'(', ')', '-', ','})

def _canonical_metric_name(name: str) -> str:
    key = _norm_metric_key(name)
    return _CANONICAL_METRIC_BY_KEY.get(key, str(name))

def _build_shared_eval_plan(
    data_variant: str = '1phase_hourly',
    max_files: int | None = None,
    seed: int = 42,
    config_path: Path = CONFIG_PATH,
) -> dict:
    if data_variant not in DATA_SOURCES:
        raise ValueError(f"Unknown data_variant='{data_variant}'. Choose one of: {list(DATA_SOURCES.keys())}")

    data_root = DATA_SOURCES[data_variant].resolve()
    if not data_root.exists():
        raise FileNotFoundError(f'Data source folder does not exist: {data_root}')

    parquet_paths = sorted(str(p.resolve()) for p in data_root.glob('*.parquet'))
    if not parquet_paths:
        raise FileNotFoundError(f'No parquet files found in {data_root}')

    if max_files is not None and max_files > 0 and max_files < len(parquet_paths):
        rng = random.Random(seed)
        parquet_paths = sorted(rng.sample(parquet_paths, max_files))

    with open(config_path, 'r') as f:
        cfg_raw = yaml.safe_load(f) or {}

    data_cfg = cfg_raw.get('data', {}) if isinstance(cfg_raw, dict) else {}
    dataloader_cfg = data_cfg.get('dataloader', {}) if isinstance(data_cfg, dict) else {}
    phased_cfg = data_cfg.get('phased', {}) if isinstance(data_cfg, dict) else {}

    plan = {
        'config_path': str(Path(config_path).resolve()),
        'data_variant': data_variant,
        'data_root': str(data_root),
        'parquet_paths': parquet_paths,
        'batch_size': int(dataloader_cfg.get('batch_size', 2048)),
        'eval_batches': int(phased_cfg.get('eval_batches', 5)),
        'min_buffer_size': int(phased_cfg.get('min_buffer_size', 2000)),
        'replay_buffer_capacity': int(phased_cfg.get('replay_buffer_capacity', 10000)),
    }
    return plan

def _find_checkpoint(model_dir: Path) -> Path | None:
    candidates = sorted(model_dir.glob('*_final.pt'))
    if candidates:
        return candidates[-1]
    candidates = sorted(model_dir.glob('*_best.pt'))
    if candidates:
        return candidates[-1]
    candidates = sorted(model_dir.glob('*.pt'))
    return candidates[-1] if candidates else None

def _metric_goal(metric_name: str) -> str:
    if metric_name == 'TSTR (Train on Synthetic, Test on Real)':
        return 'target=1'
    if 'Log-Likelihood' in metric_name or metric_name == 'Visual Evaluations':
        return 'higher'
    return 'lower'

def _is_better(metric_name: str, model_value, baseline_value):
    try:
        m = float(model_value)
        b = float(baseline_value)
    except Exception:
        return None
    if not np.isfinite(m) or not np.isfinite(b):
        return None
    goal = _metric_goal(metric_name)
    if goal == 'higher':
        return m > b
    if goal == 'target=1':
        return abs(m - 1.0) < abs(b - 1.0)
    return m < b

def _fmt_val(v) -> str:
    if v is None:
        return '-'
    if isinstance(v, str):
        return v
    if isinstance(v, (int, float, np.floating)):
        return f'{float(v):.2f}'
    return str(v)

def _render_table(rows, headers):
    if _tabulate is not None:
        return _tabulate(rows, headers=headers, tablefmt='grid')
    matrix = [headers] + [[str(c) for c in r] for r in rows]
    widths = [max(len(row[i]) for row in matrix) for i in range(len(headers))]
    sep = '+' + '+'.join('-' * (w + 2) for w in widths) + '+'
    out = [sep]
    out.append('| ' + ' | '.join(headers[i].ljust(widths[i]) for i in range(len(headers))) + ' |')
    out.append(sep)
    for row in rows:
        out.append('| ' + ' | '.join(str(row[i]).ljust(widths[i]) for i in range(len(headers))) + ' |')
    out.append(sep)
    return '\n'.join(out)

def _extract_metric_map(data: dict) -> dict:
    """Accept both formats: {'standalone_metrics': {...}} and top-level scalar metric dict."""
    if not isinstance(data, dict):
        return {}

    if isinstance(data.get('standalone_metrics'), dict) and data['standalone_metrics']:
        metric_map = {}
        for metric, payload in data['standalone_metrics'].items():
            canonical = _canonical_metric_name(metric)
            if isinstance(payload, dict):
                metric_map[canonical] = payload.get('value', None)
            else:
                metric_map[canonical] = payload
        return metric_map

    metric_map = {}
    ignored = {'standalone_metrics', 'results', 'metadata', 'run_info'}
    for key, value in data.items():
        if key in ignored:
            continue
        canonical = _canonical_metric_name(key)
        if isinstance(value, (int, float, np.floating)):
            metric_map[canonical] = float(value)
        elif isinstance(value, str) and value.upper() == 'ERROR':
            metric_map[canonical] = 'ERROR'
        elif isinstance(value, dict) and 'value' in value:
            metric_map[canonical] = value.get('value')

    if metric_map:
        return metric_map

    if isinstance(data.get('results'), dict):
        for key, value in data['results'].items():
            canonical = _canonical_metric_name(key)
            if isinstance(value, (int, float, np.floating)) or (isinstance(value, str) and value.upper() == 'ERROR'):
                metric_map[canonical] = value
    return metric_map

def _load_baseline_metric_map_from_file(path: Path) -> dict:
    if not path.exists():
        return {}
    with open(path, 'r') as f:
        return _extract_metric_map(json.load(f))

def _baseline_cache_path(shared_plan: dict | None) -> Path:
    variant = 'unknown' if not shared_plan else str(shared_plan.get('data_variant', 'unknown'))
    return BASELINE_CACHE_DIR / f'eval_results_baseline_mean_{variant}.json'

def _create_shared_baseline(shared_plan: dict) -> dict:
    """Create baseline file by running one supported model eval if no baseline exists yet."""
    preferred_models = [
        'SetGuideVAE_DictionaryCoder_20260416_103152',
        'CVAE_DictionaryCoder_20260416_082400',
        'CVAE_GaussianCoder_20260416_085358',
    ]
    for model_name in preferred_models:
        model_dir = HPC_RUNS_DIR / model_name
        ckpt = _find_checkpoint(model_dir)
        if ckpt is None:
            continue
        try:
            run_checkpoint_evaluation(
                checkpoint_path=str(ckpt),
                config_path=str(CONFIG_PATH),
                device='cpu',
                include_visualization=False,
                output_dir=str(model_dir),
                shared_parquet_paths=shared_plan['parquet_paths'],
                strict_shared_eval=True,
                eval_seed=42,
                override_batch_size=shared_plan['batch_size'],
                override_eval_batches=shared_plan['eval_batches'],
                override_min_buffer_size=shared_plan['min_buffer_size'],
                override_replay_buffer_capacity=shared_plan['replay_buffer_capacity'],
            )
            baseline_local = model_dir / 'eval_results_baseline_mean.json'
            if baseline_local.exists():
                cache_path = _baseline_cache_path(shared_plan)
                cache_path.write_text(baseline_local.read_text())
                return _load_baseline_metric_map_from_file(cache_path)
        except Exception as e:
            print(f'Baseline creation attempt failed for {model_name}: {e}')
            continue
    return {}

def _get_or_create_baseline_metric_map(model_dir: Path, shared_plan: dict | None) -> tuple[dict, str]:
    """Load local baseline; else load shared cache; else create and cache baseline."""
    local_candidates = [
        model_dir / 'eval_results_baseline_mean.json',
        model_dir / 'eval_results_baseline.json',
        model_dir / 'baseline_eval_results.json',
    ]
    for p in local_candidates:
        metrics = _load_baseline_metric_map_from_file(p)
        if metrics:
            return metrics, f'local:{p.name}'

    cache_path = _baseline_cache_path(shared_plan)
    cache_metrics = _load_baseline_metric_map_from_file(cache_path)
    if cache_metrics:
        return cache_metrics, f'shared-cache:{cache_path.name}'

    if shared_plan is not None:
        created = _create_shared_baseline(shared_plan)
        if created:
            return created, f'created:{cache_path.name}'

    return {}, 'missing'

def _display_eval_results(eval_json_path: Path, shared_plan: dict | None = None):
    if not eval_json_path.exists():
        print(f'No eval results found at {eval_json_path}')
        return None
    with open(eval_json_path, 'r') as f:
        data = json.load(f)
    print(f'Loaded: {eval_json_path}')

    metrics = _extract_metric_map(data)
    if not metrics:
        print('No scalar metrics found in eval_results.json')
        return data

    baseline_metrics, baseline_source = _get_or_create_baseline_metric_map(eval_json_path.parent, shared_plan)

    rows = []
    for metric in sorted(metrics.keys()):
        model_value = metrics.get(metric, None)
        base_value = baseline_metrics.get(metric, None)

        goal = _metric_goal(metric)
        delta = '-'
        better = '-'

        if isinstance(model_value, (int, float, np.floating)) and isinstance(base_value, (int, float, np.floating)):
            if goal == 'target=1':
                delta_num = abs(float(base_value) - 1.0) - abs(float(model_value) - 1.0)
            else:
                delta_num = float(model_value) - float(base_value)
            delta = f'{delta_num:+.2f}'
            better_bool = _is_better(metric, model_value, base_value)
            better = 'yes' if better_bool else ('no' if better_bool is not None else '-')

        rows.append([
            metric,
            goal,
            _fmt_val(model_value) if model_value is not None else 'ERROR',
            _fmt_val(base_value),
            delta,
            better,
        ])

    print('\nEvaluation summary')
    headers = ['Metric', 'Goal', 'Model', 'Mean Baseline', 'Delta', 'Better vs Baseline']
    print(_render_table(rows, headers))

    if baseline_metrics:
        print(f'Baseline source: {baseline_source}')
    else:
        print('Baseline could not be loaded/created for this run.')
    return data

def _list_visual_artifacts(model_dir: Path):
    image_patterns = [
        'training_progress.png',
        'synthetic_samples.png',
        'synthetic_recreations_single_phase.png',
        'reconstruction_single_phase.png',
        'PCA_latent_space.png',
        'tSNE_latent_space.png',
        'sigma_mapper.png',
        'user_profiles.png',
    ]
    pdf_patterns = [
        'training_progress.pdf',
        'synthetic_samples.pdf',
        'synthetic_recreations_single_phase.pdf',
        'reconstruction_single_phase.pdf',
        'PCA_latent_space.pdf',
        'tSNE_latent_space.pdf',
        'sigma_mapper.pdf',
        'user_profiles.pdf',
    ]

    image_files = [model_dir / p for p in image_patterns if (model_dir / p).exists()]
    pdf_files = [model_dir / p for p in pdf_patterns if (model_dir / p).exists()]

    if image_files or pdf_files:
        print('Visualization artifacts:')
        for p in image_files + pdf_files:
            print(f'  - {p.relative_to(PROJECT_ROOT)}')
    else:
        print('No known visualization artifacts found yet.')

    return image_files, pdf_files

def _show_visual_artifacts(model_dir: Path):
    image_files, pdf_files = _list_visual_artifacts(model_dir)

    if image_files:
        display(Markdown('#### Plots in notebook'))
        for p in image_files:
            display(Markdown(f"**{p.name}**"))
            try:
                img = plt.imread(str(p))
                plt.figure(figsize=(14, 5))
                plt.imshow(img)
                plt.axis('off')
                plt.tight_layout()
                plt.show()
            except Exception as e:
                print(f'Could not plot {p.name}: {e}')

    if pdf_files:
        display(Markdown('#### PDF plots in notebook'))
        for p in pdf_files:
            display(Markdown(f"**{p.name}**"))
            try:
                display(IFrame(src=p.as_uri(), width='100%', height=650))
            except Exception as e:
                print(f'Could not render PDF {p.name}: {e}')

def evaluate_model_run(model_name: str, shared_plan: dict, rerun_if_supported: bool = True):
    model_dir = HPC_RUNS_DIR / model_name
    if not model_dir.exists():
        print(f'Model directory missing: {model_dir}')
        return None

    print(f'\n=== {model_name} ===')
    checkpoint = _find_checkpoint(model_dir)
    if checkpoint is None:
        print('No checkpoint found, trying to show existing eval results only.')
        data = _display_eval_results(model_dir / 'eval_results.json', shared_plan=shared_plan)
        _show_visual_artifacts(model_dir)
        return data

    if rerun_if_supported:
        try:
            run_checkpoint_evaluation(
                checkpoint_path=str(checkpoint),
                config_path=str(CONFIG_PATH),
                device='cpu',
                include_visualization=True,
                output_dir=str(model_dir),
                shared_parquet_paths=shared_plan['parquet_paths'],
                strict_shared_eval=True,
                eval_seed=42,
                override_batch_size=shared_plan['batch_size'],
                override_eval_batches=shared_plan['eval_batches'],
                override_min_buffer_size=shared_plan['min_buffer_size'],
                override_replay_buffer_capacity=shared_plan['replay_buffer_capacity'],
            )
        except Exception as e:
            msg = str(e)
            if 'Error(s) in loading state_dict for AutoRegressiveTransformerVAE' in msg:
                print('Rerun skipped for ART: checkpoint weights do not match current model code/hyperparameters.')
                print('Using existing eval results and plots from the model folder instead.')
            else:
                print(f'Rerun not completed for this model (fallback to existing results): {e}')

    data = _display_eval_results(model_dir / 'eval_results.json', shared_plan=shared_plan)
    _show_visual_artifacts(model_dir)
    return data

print('Helper setup complete.')
print(f'Project root: {PROJECT_ROOT}')
print(f'Current working directory: {Path.cwd()}')
print(f'HPC runs folder exists: {HPC_RUNS_DIR.exists()}')
print(f'Available data variants: {list(DATA_SOURCES.keys())}')

Helper setup complete.
Project root: /Users/clarareginekold/Desktop/dtu/THESIS/rasmus-clara-synthetic-data
Current working directory: /Users/clarareginekold/Desktop/dtu/THESIS/rasmus-clara-synthetic-data
HPC runs folder exists: True
Available data variants: ['1phase_hourly', '3phase_15min']


In [57]:
# Baseline + visualization + hyperparameter display overrides.
# Baseline priority: local file -> shared cache -> global baseline artifacts, then fill missing metrics from fleet mean.

def _find_global_baseline_files() -> list[Path]:
    candidates = []
    models_root = PROJECT_ROOT / 'models'
    if models_root.exists():
        candidates.extend(models_root.glob('**/eval_results_baseline_mean.json'))
        candidates.extend(models_root.glob('**/eval_results_baseline.json'))
        candidates.extend(models_root.glob('**/baseline_eval_results.json'))
    return sorted({p.resolve() for p in candidates if p.exists()}, key=lambda p: p.stat().st_mtime, reverse=True)

def _build_hpc_fleet_mean_baseline() -> dict:
    values_by_metric = {}
    for model_name in MODEL_RUNS:
        eval_path = HPC_RUNS_DIR / model_name / 'eval_results.json'
        if not eval_path.exists():
            continue
        try:
            with open(eval_path, 'r') as f:
                data = json.load(f)
            metric_map = _extract_metric_map(data)
        except Exception:
            continue

        for metric, value in metric_map.items():
            if isinstance(value, (int, float, np.floating)) and np.isfinite(float(value)):
                values_by_metric.setdefault(metric, []).append(float(value))

    return {metric: float(np.mean(vals)) for metric, vals in values_by_metric.items() if vals}

def _merge_with_fleet_mean(primary: dict) -> tuple[dict, int]:
    fleet_mean = _build_hpc_fleet_mean_baseline()
    merged = dict(primary or {})
    fill_count = 0
    for metric, value in fleet_mean.items():
        if metric not in merged or merged.get(metric) is None:
            merged[metric] = value
            fill_count += 1
    return merged, fill_count

def _get_or_create_baseline_metric_map(model_dir: Path, shared_plan: dict | None) -> tuple[dict, str]:
    """Load baseline from local/cache/global files and fill missing keys from fleet mean."""
    local_candidates = [
        model_dir / 'eval_results_baseline_mean.json',
        model_dir / 'eval_results_baseline.json',
        model_dir / 'baseline_eval_results.json',
    ]
    for p in local_candidates:
        metrics = _load_baseline_metric_map_from_file(p)
        if metrics:
            merged, n_fill = _merge_with_fleet_mean(metrics)
            src = f'local:{p.name}'
            if n_fill > 0:
                src += f'+fleet-mean-fill({n_fill})'
            return merged, src

    cache_path = _baseline_cache_path(shared_plan)
    cache_metrics = _load_baseline_metric_map_from_file(cache_path)
    if cache_metrics:
        merged, n_fill = _merge_with_fleet_mean(cache_metrics)
        src = f'shared-cache:{cache_path.name}'
        if n_fill > 0:
            src += f'+fleet-mean-fill({n_fill})'
        return merged, src

    for p in _find_global_baseline_files():
        metrics = _load_baseline_metric_map_from_file(p)
        if not metrics:
            continue
        try:
            cache_path.write_text(p.read_text())
            src = f'global-cached:{p.relative_to(PROJECT_ROOT)}'
        except Exception:
            src = f'global:{p}'
        merged, n_fill = _merge_with_fleet_mean(metrics)
        if n_fill > 0:
            src += f'+fleet-mean-fill({n_fill})'
        return merged, src

    fleet_only = _build_hpc_fleet_mean_baseline()
    if fleet_only:
        return fleet_only, 'fleet-mean-only'

    return {}, 'missing'

def _load_training_config_used(model_dir: Path) -> tuple[dict, str]:
    cfg_path = model_dir / 'training_config_used.yaml'
    if not cfg_path.exists():
        return {}, 'missing'
    try:
        with open(cfg_path, 'r') as f:
            cfg = yaml.safe_load(f) or {}
        return cfg if isinstance(cfg, dict) else {}, str(cfg_path.relative_to(PROJECT_ROOT))
    except Exception:
        return {}, str(cfg_path.relative_to(PROJECT_ROOT))

def _flatten_tags(tags):
    if isinstance(tags, list):
        return ', '.join(str(t) for t in tags)
    return str(tags) if tags is not None else '-'

def _print_model_hyperparameters(model_dir: Path):
    cfg, cfg_src = _load_training_config_used(model_dir)
    if not cfg:
        print('Hyperparameters: training_config_used.yaml not found or unreadable.')
        return

    model = cfg.get('model', {}) if isinstance(cfg.get('model', {}), dict) else {}
    training = cfg.get('training', {}) if isinstance(cfg.get('training', {}), dict) else {}
    data = cfg.get('data', {}) if isinstance(cfg.get('data', {}), dict) else {}
    dataloader = data.get('dataloader', {}) if isinstance(data.get('dataloader', {}), dict) else {}
    window = data.get('window', {}) if isinstance(data.get('window', {}), dict) else {}
    prior = model.get('prior', {}) if isinstance(model.get('prior', {}), dict) else {}
    encoder = model.get('encoder', {}) if isinstance(model.get('encoder', {}), dict) else {}
    decoder = model.get('decoder', {}) if isinstance(model.get('decoder', {}), dict) else {}
    lr_scheduler = training.get('lr_scheduler', {}) if isinstance(training.get('lr_scheduler', {}), dict) else {}
    conditioning = cfg.get('conditioning', {}) if isinstance(cfg.get('conditioning', {}), dict) else {}

    items = [
        ('config', cfg_src),
        ('model.type', model.get('type')),
        ('model.latent_dim', model.get('latent_dim')),
        ('model.encoder.type', encoder.get('type')),
        ('model.decoder.type', decoder.get('type')),
        ('model.prior.type', prior.get('type')),
        ('model.prior.dictionary_size', prior.get('dictionary_size')),
        ('training.optimizer', training.get('optimizer')),
        ('training.learning_rate', training.get('learning_rate')),
        ('training.weight_decay', training.get('weight_decay')),
        ('training.num_epochs', training.get('num_epochs')),
        ('training.beta', training.get('beta')),
        ('training.weighted_loss', training.get('weighted_loss')),
        ('training.batch_size', dataloader.get('batch_size')),
        ('training.window.input_len', window.get('input_len')),
        ('training.window.stride', window.get('stride')),
        ('training.lr_scheduler', lr_scheduler.get('scheduler_type') if lr_scheduler.get('enabled') else 'disabled'),
        ('conditioning.tags', _flatten_tags(conditioning.get('tags'))),
    ]

    model_type = str(model.get('type', '')).lower()
    if model_type == 'art':
        art = model.get('art', {}) if isinstance(model.get('art', {}), dict) else {}
        items.extend([
            ('model.art.num_bins', art.get('num_bins')),
            ('model.art.d_model', art.get('d_model')),
            ('model.art.transformer_num_heads', art.get('transformer_num_heads')),
            ('model.art.transformer_num_layers', art.get('transformer_num_layers')),
        ])
    elif model_type == 'diffusion':
        diff = model.get('diffusion', {}) if isinstance(model.get('diffusion', {}), dict) else {}
        items.extend([
            ('model.diffusion.num_timesteps', diff.get('num_timesteps')),
            ('model.diffusion.schedule_type', diff.get('schedule_type')),
            ('model.diffusion.base_channels', diff.get('base_channels')),
            ('model.diffusion.num_blocks', diff.get('num_blocks')),
        ])

    rows = [[k, str(v)] for k, v in items if v is not None]
    print('Hyperparameters used')
    print(_render_table(rows, headers=['Key', 'Value']))

def _show_visual_artifacts(model_dir: Path):
    image_files, _ = _list_visual_artifacts(model_dir)

    if image_files:
        display(Markdown('#### Plots in notebook (PNG only)'))
        for p in image_files:
            display(Markdown(f"**{p.name}**"))
            try:
                img = plt.imread(str(p))
                plt.figure(figsize=(14, 5))
                plt.imshow(img)
                plt.axis('off')
                plt.tight_layout()
                plt.show()
            except Exception as e:
                print(f'Could not plot {p.name}: {e}')

def evaluate_model_run(model_name: str, shared_plan: dict, rerun_if_supported: bool = True):
    model_dir = HPC_RUNS_DIR / model_name
    if not model_dir.exists():
        print(f'Model directory missing: {model_dir}')
        return None

    print(f'\n=== {model_name} ===')
    _print_model_hyperparameters(model_dir)

    checkpoint = _find_checkpoint(model_dir)
    if checkpoint is None:
        print('No checkpoint found, trying to show existing eval results only.')
        data = _display_eval_results(model_dir / 'eval_results.json', shared_plan=shared_plan)
        _show_visual_artifacts(model_dir)
        return data

    if rerun_if_supported:
        try:
            run_checkpoint_evaluation(
                checkpoint_path=str(checkpoint),
                config_path=str(CONFIG_PATH),
                device='cpu',
                include_visualization=True,
                output_dir=str(model_dir),
                shared_parquet_paths=shared_plan['parquet_paths'],
                strict_shared_eval=True,
                eval_seed=42,
                override_batch_size=shared_plan['batch_size'],
                override_eval_batches=shared_plan['eval_batches'],
                override_min_buffer_size=shared_plan['min_buffer_size'],
                override_replay_buffer_capacity=shared_plan['replay_buffer_capacity'],
            )
        except Exception as e:
            msg = str(e)
            if 'Unsupported model class for phased checkpoint evaluation' in msg:
                print('Rerun skipped: model class is not supported by load_checkpoint phased evaluation.')
                print('Using existing eval results and plots from the model folder instead.')
            elif 'Error(s) in loading state_dict for AutoRegressiveTransformerVAE' in msg:
                print('Rerun skipped for ART: checkpoint weights do not match current model code/hyperparameters.')
                print('Using existing eval results and plots from the model folder instead.')
            else:
                print(f'Rerun not completed for this model (fallback to existing results): {e}')

    data = _display_eval_results(model_dir / 'eval_results.json', shared_plan=shared_plan)
    _show_visual_artifacts(model_dir)
    return data

print('Overrides loaded: baseline from files + fleet-mean fill, PDF inline plotting disabled, hyperparameter print enabled.')

Overrides loaded: baseline from files + fleet-mean fill, PDF inline plotting disabled, hyperparameter print enabled.


In [58]:
# Override evaluate_model_run to use each run's saved config and auto-switch SetGuide to 3-phase data.
def evaluate_model_run(model_name: str, shared_plan: dict, rerun_if_supported: bool = True):
    model_dir = HPC_RUNS_DIR / model_name
    if not model_dir.exists():
        print(f'Model directory missing: {model_dir}')
        return None

    print(f'\n=== {model_name} ===')
    _print_model_hyperparameters(model_dir)

    run_config_path = model_dir / 'training_config_used.yaml'
    effective_config_path = run_config_path if run_config_path.exists() else CONFIG_PATH
    effective_plan = shared_plan

    # SetGuideVAE runs here were trained/evaluated on 3-phase data.
    if model_name.startswith('SetGuideVAE_') and str(shared_plan.get('data_variant', '')) != '3phase_15min':
        try:
            effective_plan = _build_shared_eval_plan(
                data_variant='3phase_15min',
                max_files=None,
                seed=42,
                config_path=effective_config_path,
            )
            print("Using SetGuide-specific shared plan: 3phase_15min")
        except Exception as e:
            print(f"Could not build SetGuide 3-phase plan, keeping current shared plan: {e}")

    checkpoint = _find_checkpoint(model_dir)
    if checkpoint is None:
        print('No checkpoint found, trying to show existing eval results only.')
        data = _display_eval_results(model_dir / 'eval_results.json', shared_plan=effective_plan)
        _show_visual_artifacts(model_dir)
        return data

    if rerun_if_supported:
        try:
            run_checkpoint_evaluation(
                checkpoint_path=str(checkpoint),
                config_path=str(effective_config_path),
                device='cpu',
                include_visualization=True,
                output_dir=str(model_dir),
                shared_parquet_paths=effective_plan['parquet_paths'],
                strict_shared_eval=True,
                eval_seed=42,
                override_batch_size=effective_plan['batch_size'],
                override_eval_batches=effective_plan['eval_batches'],
                override_min_buffer_size=effective_plan['min_buffer_size'],
                override_replay_buffer_capacity=effective_plan['replay_buffer_capacity'],
            )
        except Exception as e:
            msg = str(e)
            if 'Unsupported model class for phased checkpoint evaluation' in msg:
                print('Rerun skipped: model class is not supported by load_checkpoint phased evaluation.')
                print('Using existing eval results and plots from the model folder instead.')
            elif 'Error(s) in loading state_dict for AutoRegressiveTransformerVAE' in msg:
                print('Rerun skipped for ART: checkpoint weights do not match current model code/hyperparameters.')
                print('Using existing eval results and plots from the model folder instead.')
            elif 'Error(s) in loading state_dict for SetGuideVAE' in msg:
                print('Rerun skipped for SetGuideVAE: checkpoint does not match current code signature.')
                print('Using existing eval artifacts from the model folder instead.')
            else:
                print(f'Rerun not completed for this model (fallback to existing results): {e}')

    data = _display_eval_results(model_dir / 'eval_results.json', shared_plan=effective_plan)
    _show_visual_artifacts(model_dir)
    return data

print('evaluate_model_run override loaded: per-run config + SetGuide 3-phase plan handling enabled.')

evaluate_model_run override loaded: per-run config + SetGuide 3-phase plan handling enabled.


In [59]:
# Override _display_eval_results to include selected per-condition numeric rows in the summary table.
_PER_CONDITION_INCLUDE_METRICS = {'mmd', 'tstr'}
_REMOVE_STANDALONE_METRICS = {'Classical Statistics', 'Per-Condition Evaluation'}

def _coerce_condition_scalar(value):
    if isinstance(value, (int, float, np.floating)) and np.isfinite(float(value)):
        return float(value)
    if isinstance(value, dict):
        if isinstance(value.get('fd', None), (int, float, np.floating)):
            return float(value['fd'])
        diffs = value.get('differences', None)
        if isinstance(diffs, dict) and isinstance(diffs.get('overall_mean_abs_diff', None), (int, float, np.floating)):
            return float(diffs['overall_mean_abs_diff'])
        if isinstance(value.get('value', None), (int, float, np.floating)):
            return float(value['value'])
        for v in value.values():
            if isinstance(v, (int, float, np.floating)) and np.isfinite(float(v)):
                return float(v)
    return None

def _extract_per_condition_metric_map(model_dir: Path) -> dict:
    condition_path = model_dir / 'eval_results_condition.json'
    if not condition_path.exists():
        return {}
    try:
        with open(condition_path, 'r') as f:
            data = json.load(f)
    except Exception:
        return {}

    out = {}
    for metric_name, by_condition in data.items():
        if metric_name not in _PER_CONDITION_INCLUDE_METRICS:
            continue
        if not isinstance(by_condition, dict):
            continue
        for condition_name, by_bucket in by_condition.items():
            if not isinstance(by_bucket, dict):
                continue
            for bucket, payload in by_bucket.items():
                scalar = _coerce_condition_scalar(payload)
                if scalar is None:
                    continue
                label = f"Per-Condition/{metric_name}/{condition_name}={bucket}"
                out[label] = scalar
    return out

def _metric_goal_for_row(metric_name: str) -> str:
    m = str(metric_name).lower()
    if m.startswith('per-condition/') and '/tstr/' in m:
        return 'target=1'
    return _metric_goal(metric_name)

def _fix_special_baseline_values(metrics: dict, baseline_metrics: dict, baseline_source: str) -> tuple[dict, str]:
    fixed = dict(baseline_metrics)
    source = baseline_source
    fleet = _build_hpc_fleet_mean_baseline()

    agg_key = 'Aggregated Profiles (Flexibility Distance)'
    agg_base = fixed.get(agg_key, None)
    agg_fleet = fleet.get(agg_key, None)
    if isinstance(agg_base, (int, float, np.floating)) and float(agg_base) == 0.0:
        if isinstance(agg_fleet, (int, float, np.floating)) and np.isfinite(float(agg_fleet)) and float(agg_fleet) > 0.0:
            fixed[agg_key] = float(agg_fleet)
            source = f'{source}+agg-from-fleet-mean'

    return fixed, source

def _display_eval_results(eval_json_path: Path, shared_plan: dict | None = None):
    if not eval_json_path.exists():
        print(f'No eval results found at {eval_json_path}')
        return None
    with open(eval_json_path, 'r') as f:
        data = json.load(f)
    print(f'Loaded: {eval_json_path}')

    metrics = _extract_metric_map(data)
    for rm in _REMOVE_STANDALONE_METRICS:
        metrics.pop(rm, None)

    per_condition_metrics = _extract_per_condition_metric_map(eval_json_path.parent)
    metrics = {**metrics, **per_condition_metrics}

    if not metrics:
        print('No scalar metrics found in eval results files')
        return data

    baseline_metrics, baseline_source = _get_or_create_baseline_metric_map(eval_json_path.parent, shared_plan)
    baseline_metrics, baseline_source = _fix_special_baseline_values(metrics, baseline_metrics, baseline_source)

    rows = []
    for metric in sorted(metrics.keys()):
        model_value = metrics.get(metric, None)
        base_value = baseline_metrics.get(metric, None)

        goal = _metric_goal_for_row(metric)
        delta = '-'
        better = '-'

        if isinstance(model_value, (int, float, np.floating)) and isinstance(base_value, (int, float, np.floating)):
            if goal == 'target=1':
                delta_num = abs(float(base_value) - 1.0) - abs(float(model_value) - 1.0)
            else:
                delta_num = float(model_value) - float(base_value)
            delta = f'{delta_num:+.2f}'
            better_bool = _is_better(metric if goal != 'target=1' else 'TSTR (Train on Synthetic, Test on Real)', model_value, base_value)
            better = 'yes' if better_bool else ('no' if better_bool is not None else '-')

        rows.append([
            metric,
            goal,
            _fmt_val(model_value) if model_value is not None else 'ERROR',
            _fmt_val(base_value),
            delta,
            better,
        ])

    print('\nEvaluation summary')
    headers = ['Metric', 'Goal', 'Model', 'Mean Baseline', 'Delta', 'Better vs Baseline']
    print(_render_table(rows, headers))

    if baseline_metrics:
        print(f'Baseline source: {baseline_source}')
    else:
        print('Baseline could not be loaded/created for this run.')
    return data

In [74]:
# Override baseline source: use mean(training) as generated data for all metrics, with per-model synthetic count.
import torch
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import yaml
from datetime import datetime, timezone
from data.dataloader_phased import WindowSpec, PhasedReplayDataloaders
from evaluation.eval_framework import (
    ReconstructionMSE,
    ReconstructionRMSE,
    ReconstructionLogLikelihood,
    FlexibilityDistance,
    MaximumMeanDiscrepancy,
    SetMaximumMeanDiscrepancy,
    ACFDistribution,
    ClassicalStatistics,
    TSTR,
    AggregatedProfiles,
    _sanitize_numeric_array,
)

_MEAN_BASELINE_CACHE = {}
_BASELINE_SYNTHETIC_SAMPLE_COUNT = 1000
_HOURLY_TRAIN_EVAL_CACHE = {}
_SHARED_MEAN_BASELINE_CACHE = {}
_SKIP_BASELINE_TSTR = False

def _window_size_from_variant(variant: str) -> int:
    return 24 if str(variant) == '1phase_hourly' else 96

def _build_hourly_train_eval_arrays(shared_plan: dict):
    """Build [N, 24] windows for hourly files using parquet columns (not filename meter parsing)."""
    parquet_paths = list(shared_plan.get('parquet_paths', []))
    if not parquet_paths:
        raise RuntimeError('No parquet paths provided in shared plan.')

    target_eval = max(1, int(shared_plan['batch_size']) * int(shared_plan['eval_batches']))
    target_train = target_eval
    target_total = target_train + target_eval
    cache_key = (tuple(parquet_paths), int(target_train), int(target_eval))
    cached = _HOURLY_TRAIN_EVAL_CACHE.get(cache_key, None)
    if cached is not None:
        return cached

    # Balanced and still bounded: sample across all monthly files.
    target_windows = max(target_total * 2, target_total)
    n_files = max(1, len(parquet_paths))
    per_file_quota = max(1, int(np.ceil(target_windows / n_files)))
    max_rows_per_file = per_file_quota * 24 * 3

    windows = []
    meter_ids = []
    ts_end = []

    consumption_candidates = [
        'hourly_energy_consumption',
        'active_power_p14_l1',
        'active_power_p14_l1_w',
    ]
    meter_candidates = ['meter_number', 'meter_id', 'meter']
    timestamp_candidates = ['timestamp_dst', 'timestamp', 'timestamp_utc']

    for path in parquet_paths:
        file_added = 0
        try:
            df = pq.read_table(path).to_pandas()
        except Exception:
            continue
        if df is None or df.empty:
            continue

        # Hourly parquet can store meter/timestamp in index; expose them as columns.
        if isinstance(df.index, pd.MultiIndex) or df.index.name is not None:
            df = df.reset_index()

        consumption_col = next((c for c in consumption_candidates if c in df.columns), None)
        if consumption_col is None:
            numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
            if not numeric_cols:
                continue
            consumption_col = numeric_cols[0]

        meter_col = next((c for c in meter_candidates if c in df.columns), None)
        ts_col = next((c for c in timestamp_candidates if c in df.columns), None)

        if meter_col is None:
            df = df.copy()
            df['_meter_tmp'] = 0
            meter_col = '_meter_tmp'
        if ts_col is None:
            # deterministic synthetic timestamps if missing
            df = df.copy()
            df['_ts_tmp'] = np.arange(len(df), dtype=np.int64)
            ts_col = '_ts_tmp'

        work = df[[meter_col, ts_col, consumption_col]].copy()
        work[consumption_col] = pd.to_numeric(work[consumption_col], errors='coerce')
        work = work.dropna(subset=[consumption_col])
        if work.empty:
            continue

        work[ts_col] = pd.to_datetime(work[ts_col], errors='coerce', utc=True)
        work = work.dropna(subset=[ts_col])
        work = work.sort_values([meter_col, ts_col])

        for meter, grp in work.groupby(meter_col):
            vals = grp[consumption_col].to_numpy(dtype=np.float64)
            ts_vals = grp[ts_col].to_numpy()
            if len(vals) < 24:
                continue
            for start in range(0, len(vals) - 24 + 1, 24):
                end = start + 24
                windows.append(vals[start:end])
                meter_ids.append(int(meter) if str(meter).isdigit() else 0)
                ts = pd.Timestamp(ts_vals[end - 1]).timestamp()
                ts_end.append(int(ts))
                file_added += 1
                if file_added >= per_file_quota:
                    break
            if file_added >= per_file_quota:
                break

    # Collect across all files, then sample/shuffle deterministically.

    if len(windows) < 2:
        raise RuntimeError('Could not build enough hourly windows from parquet files for baseline.')

    arr = np.asarray(windows, dtype=np.float64)
    meter_ids = np.asarray(meter_ids, dtype=np.int64)
    ts_end = np.asarray(ts_end, dtype=np.int64)

    # Keep runtime bounded.
    if len(arr) > target_total * 4:
        rng = np.random.default_rng(42)
        keep_idx = np.sort(rng.choice(len(arr), size=target_total * 4, replace=False))
        arr = arr[keep_idx]
        meter_ids = meter_ids[keep_idx]
        ts_end = ts_end[keep_idx]

    # Deterministic shuffle before splitting so eval/train each cover multiple months.
    rng = np.random.default_rng(42)
    perm = rng.permutation(len(arr))
    arr = arr[perm]
    meter_ids = meter_ids[perm]
    ts_end = ts_end[perm]

    n_train = min(target_train, max(1, len(arr) // 2))
    n_eval = min(target_eval, len(arr) - n_train)
    if n_eval <= 0:
        n_eval = min(target_eval, len(arr))
        n_train = max(1, len(arr) - n_eval)

    train_flat = arr[:n_train]
    eval_flat = arr[n_train:n_train + n_eval]
    eval_meter_ids = meter_ids[n_train:n_train + n_eval]
    eval_timestamps = ts_end[n_train:n_train + n_eval]

    if len(eval_flat) == 0:
        raise RuntimeError('Hourly baseline eval split is empty.')

    out = (train_flat, eval_flat, eval_meter_ids, eval_timestamps)
    _HOURLY_TRAIN_EVAL_CACHE[cache_key] = out
    return out

def _build_train_eval_arrays(shared_plan: dict):
    """Build train/eval windows and metadata from the shared parquet plan."""
    variant = str(shared_plan.get('data_variant', '1phase_hourly'))
    if variant == '1phase_hourly':
        return _build_hourly_train_eval_arrays(shared_plan)

    window_size = _window_size_from_variant(variant)

    replay = PhasedReplayDataloaders(
        sm_files=list(shared_plan['parquet_paths']),
        window_spec=WindowSpec(input_len=window_size, stride=window_size),
        batch_size=int(shared_plan['batch_size']),
        replay_buffer_capacity=int(shared_plan['replay_buffer_capacity']),
        min_buffer_size=int(shared_plan['min_buffer_size']),
        normalizer=None,
        num_workers=0,
        hourly_aggregation=(variant == '1phase_hourly'),
        aggregation_divisor=1.0,
        eval_cache_device=None,
        eval_cache_batches=int(shared_plan['eval_batches']),
        sample_random=False,
    )

    # Deterministic split between train/eval cache parts.
    torch.manual_seed(42)
    target_eval = int(shared_plan['batch_size']) * int(shared_plan['eval_batches'])
    target_train = target_eval
    train_x, train_c, _, eval_x, eval_c, _ = replay._collect_cache(target_train=target_train, target_eval=target_eval)

    if train_x is None or eval_x is None or train_x.shape[0] == 0 or eval_x.shape[0] == 0:
        # Fallback for sparse/strict phased streams (notably 3-phase): retry with relaxed cache settings.
        replay_relaxed = PhasedReplayDataloaders(
            sm_files=list(shared_plan['parquet_paths']),
            window_spec=WindowSpec(input_len=window_size, stride=window_size),
            batch_size=min(256, int(shared_plan['batch_size'])),
            replay_buffer_capacity=max(4096, int(shared_plan['replay_buffer_capacity'])),
            min_buffer_size=min(256, int(shared_plan['min_buffer_size'])),
            normalizer=None,
            num_workers=0,
            hourly_aggregation=(variant == '1phase_hourly'),
            aggregation_divisor=1.0,
            eval_cache_device=None,
            eval_cache_batches=max(2, min(4, int(shared_plan['eval_batches']))),
            sample_random=True,
        )
        target_eval_relaxed = max(64, min(1024, int(shared_plan['batch_size']) * int(shared_plan['eval_batches'])))
        target_train_relaxed = target_eval_relaxed
        train_x, train_c, _, eval_x, eval_c, _ = replay_relaxed._collect_cache(target_train=target_train_relaxed, target_eval=target_eval_relaxed)

    if train_x is None or eval_x is None or train_x.shape[0] == 0 or eval_x.shape[0] == 0:
        raise RuntimeError('Could not build non-empty train/eval arrays for live mean baseline.')

    train_flat = train_x.detach().cpu().numpy().reshape(train_x.shape[0], -1)
    eval_flat = eval_x.detach().cpu().numpy().reshape(eval_x.shape[0], -1)

    meter_ids = None
    timestamps = None
    if eval_c is not None and torch.is_tensor(eval_c):
        c = eval_c.detach().cpu()
        if c.dim() == 3 and c.shape[1] >= 2:
            meter_ids = c[:, 0, 0].numpy().astype(np.int64)
            timestamps = c[:, 1, 0].numpy().astype(np.int64)
        elif c.dim() == 2 and c.shape[1] >= 2:
            meter_ids = c[:, 0].numpy().astype(np.int64)
            timestamps = c[:, 1].numpy().astype(np.int64)

    return train_flat, eval_flat, meter_ids, timestamps

def _extract_consumption_windows(flat_data: np.ndarray, window_size: int) -> np.ndarray:
    arr = _sanitize_numeric_array(np.asarray(flat_data))
    if arr.ndim != 2:
        raise ValueError(f'Expected 2D [N, D], got {arr.shape}')
    if arr.shape[1] == window_size:
        return arr
    if arr.shape[1] % window_size != 0:
        raise ValueError(f'Cannot infer features from shape {arr.shape} with window_size={window_size}')

    n_features = arr.shape[1] // window_size
    # Flatten order from phased loader is feature-first when flattened from [F, L].
    return arr.reshape(arr.shape[0], n_features, window_size)[:, 0, :]

def _safe_metric_compute(metric, *, real_data=None, reconstructed_data=None, synthetic_data=None, output_path=None):
    if real_data is not None:
        setattr(metric, 'real_data', real_data)
    if reconstructed_data is not None:
        setattr(metric, 'reconstructed_data', reconstructed_data)
    if synthetic_data is not None:
        setattr(metric, 'synthetic_data', synthetic_data)
    if output_path is not None:
        setattr(metric, 'output_path', output_path)
    try:
        v = metric.compute()
        return float(v) if isinstance(v, (int, float, np.floating, np.integer)) else None
    except Exception:
        return None

def _nested_get(dct: dict, path: str):
    cur = dct
    for key in path.split('.'):
        if not isinstance(cur, dict) or key not in cur:
            return None
        cur = cur[key]
    return cur

def _resolve_model_generated_count(model_dir: Path | None, eval_count: int) -> tuple[int, str]:
    """Resolve synthetic generation count from each run's config, with safe fallback."""
    fallback = max(1, min(int(eval_count), int(_BASELINE_SYNTHETIC_SAMPLE_COUNT)))
    if model_dir is None:
        return fallback, 'default(no-model-dir)'

    cfg_path = model_dir / 'training_config_used.yaml'
    if not cfg_path.exists():
        return fallback, 'default(config-missing)'

    try:
        with open(cfg_path, 'r') as f:
            cfg = yaml.safe_load(f) or {}
    except Exception:
        return fallback, 'default(config-unreadable)'

    candidate_paths = [
        'evaluation.n_synthetic_samples',
        'evaluation.synthetic_samples',
        'training.n_synthetic_samples',
        'training.synthetic_samples',
        'model.n_synthetic_samples',
        'visualize.num_synthetic',
    ]

    for p in candidate_paths:
        raw = _nested_get(cfg, p)
        if raw is None:
            continue
        try:
            val = int(raw)
        except Exception:
            continue
        if val > 0:
            return max(1, min(val, int(eval_count))), f'config:{p}'

    return fallback, 'default(config-key-missing)'

def _shared_baseline_key(shared_plan: dict) -> tuple:
    return (
        str(shared_plan.get('data_variant', 'unknown')),
        int(shared_plan.get('batch_size', 0)),
        int(shared_plan.get('eval_batches', 0)),
        int(shared_plan.get('min_buffer_size', 0)),
        int(shared_plan.get('replay_buffer_capacity', 0)),
        len(shared_plan.get('parquet_paths', [])),
        bool(_SKIP_BASELINE_TSTR),
    )

def _required_per_condition_labels_all_models() -> set[str] | None:
    labels = set()
    for model_name in MODEL_RUNS:
        model_dir = HPC_RUNS_DIR / model_name
        labels_model = _required_per_condition_labels(model_dir)
        if labels_model:
            labels.update(labels_model)
    return labels if labels else None

def _required_per_condition_labels(model_dir: Path | None) -> set[str] | None:
    if model_dir is None:
        return None
    path = model_dir / 'eval_results_condition.json'
    if not path.exists():
        return None
    try:
        with open(path, 'r') as f:
            data = json.load(f)
    except Exception:
        return None

    labels = set()
    for metric_name, by_condition in (data.items() if isinstance(data, dict) else []):
        if metric_name not in {'mmd', 'tstr'}:
            continue
        if not isinstance(by_condition, dict):
            continue
        for condition_name, by_bucket in by_condition.items():
            if not isinstance(by_bucket, dict):
                continue
            for bucket in by_bucket.keys():
                labels.add(f'Per-Condition/{metric_name}/{condition_name}={bucket}')
    return labels if labels else None

def _compute_per_condition_baseline(real_cons: np.ndarray, synth_cons: np.ndarray, timestamps: np.ndarray, required_labels: set[str] | None = None) -> dict:
    out = {}
    if timestamps is None:
        return out

    n = min(len(real_cons), len(synth_cons), len(timestamps))
    if n <= 1:
        return out

    real_cons = real_cons[:n]
    synth_cons = synth_cons[:n]
    timestamps = timestamps[:n]

    dows = np.array([datetime.fromtimestamp(int(ts), tz=timezone.utc).weekday() for ts in timestamps], dtype=np.int64)
    months = np.array([datetime.fromtimestamp(int(ts), tz=timezone.utc).month for ts in timestamps], dtype=np.int64)

    if required_labels:
        day_mmd = set()
        month_mmd = set()
        day_tstr = set()
        month_tstr = set()
        for label in required_labels:
            s = str(label)
            try:
                bucket = int(s.split('=')[-1])
            except Exception:
                continue
            if s.startswith('Per-Condition/mmd/day_of_week='):
                day_mmd.add(bucket)
            elif s.startswith('Per-Condition/mmd/month_of_year='):
                month_mmd.add(bucket)
            elif s.startswith('Per-Condition/tstr/day_of_week='):
                day_tstr.add(bucket)
            elif s.startswith('Per-Condition/tstr/month_of_year='):
                month_tstr.add(bucket)
    else:
        day_mmd = set(range(7))
        month_mmd = set(range(1, 13))
        day_tstr = set(range(7))
        month_tstr = set(range(1, 13))

    # MMD per condition
    for day in sorted(day_mmd):
        mask = dows == day
        if int(np.sum(mask)) >= 2:
            mmd = MaximumMeanDiscrepancy(sigma=None, seed=42, max_samples=2000)
            val = _safe_metric_compute(mmd, real_data=real_cons[mask], synthetic_data=synth_cons[mask])
            if val is not None:
                out[f'Per-Condition/mmd/day_of_week={day}'] = val

    for month in sorted(month_mmd):
        mask = months == month
        if int(np.sum(mask)) >= 2:
            mmd = MaximumMeanDiscrepancy(sigma=None, seed=42, max_samples=2000)
            val = _safe_metric_compute(mmd, real_data=real_cons[mask], synthetic_data=synth_cons[mask])
            if val is not None:
                out[f'Per-Condition/mmd/month_of_year={month}'] = val

    # TSTR per condition
    if not _SKIP_BASELINE_TSTR:
        for day in sorted(day_tstr):
            mask = dows == day
            if int(np.sum(mask)) >= 4:
                tstr = TSTR(task='next_step', window_size=real_cons.shape[1], n_features=1, predictor_type='linear', epochs=50, random_state=42)
                val = _safe_metric_compute(tstr, real_data=real_cons[mask], synthetic_data=synth_cons[mask])
                if val is not None:
                    out[f'Per-Condition/tstr/day_of_week={day}'] = val

        for month in sorted(month_tstr):
            mask = months == month
            if int(np.sum(mask)) >= 4:
                tstr = TSTR(task='next_step', window_size=real_cons.shape[1], n_features=1, predictor_type='linear', epochs=50, random_state=42)
                val = _safe_metric_compute(tstr, real_data=real_cons[mask], synthetic_data=synth_cons[mask])
                if val is not None:
                    out[f'Per-Condition/tstr/month_of_year={month}'] = val

    return out

def _compute_mean_generated_baseline_metrics(shared_plan: dict, model_dir: Path | None = None) -> tuple[dict, str]:
    """Compute baseline for all metrics by treating mean(training) profile as generated samples."""
    variant = str(shared_plan.get('data_variant', 'unknown'))

    train_flat, eval_flat, meter_ids, timestamps = _build_train_eval_arrays(shared_plan)
    window_size = _window_size_from_variant(variant)

    mean_profile_flat = np.mean(train_flat, axis=0, keepdims=True)
    n_generated, n_generated_source = _resolve_model_generated_count(model_dir, len(eval_flat))
    n_eval = max(1, min(len(eval_flat), n_generated))

    model_key = model_dir.name if model_dir is not None else 'unknown_model'
    cache_key = (
        model_key,
        variant,
        int(shared_plan.get('batch_size', 0)),
        int(shared_plan.get('eval_batches', 0)),
        int(shared_plan.get('min_buffer_size', 0)),
        int(shared_plan.get('replay_buffer_capacity', 0)),
        len(shared_plan.get('parquet_paths', [])),
        n_generated,
    )
    if cache_key in _MEAN_BASELINE_CACHE:
        return _MEAN_BASELINE_CACHE[cache_key]

    # Sample across the full eval pool for overall metrics.
    if len(eval_flat) > n_eval:
        rng = np.random.default_rng(42)
        idx = np.sort(rng.choice(len(eval_flat), size=n_eval, replace=False))
    else:
        idx = np.arange(len(eval_flat))

    real_eval = _sanitize_numeric_array(eval_flat[idx])
    recon_baseline = np.repeat(mean_profile_flat, len(real_eval), axis=0)
    synth_baseline = recon_baseline.copy()

    # Consumption-only views for metrics that expect [N, window_size] in this project.
    real_cons = _extract_consumption_windows(real_eval, window_size)
    synth_cons = _extract_consumption_windows(synth_baseline, window_size)

    # Use full eval pool for per-condition baseline coverage so day/month keys don't drop out.
    real_eval_full = _sanitize_numeric_array(eval_flat)
    synth_full = np.repeat(mean_profile_flat, len(real_eval_full), axis=0)
    real_cons_full = _extract_consumption_windows(real_eval_full, window_size)
    synth_cons_full = _extract_consumption_windows(synth_full, window_size)
    ts_full = (timestamps if timestamps is not None and len(timestamps) == len(eval_flat) else None)

    scratch_output = str(BASELINE_CACHE_DIR / f'baseline_metric_scratch_{variant}.json')

    baseline = {}
    baseline['Reconstruction MSE'] = _safe_metric_compute(
        ReconstructionMSE(),
        real_data=real_eval,
        reconstructed_data=recon_baseline,
        output_path=scratch_output,
    )
    baseline['Reconstruction RMSE'] = _safe_metric_compute(
        ReconstructionRMSE(),
        real_data=real_eval,
        reconstructed_data=recon_baseline,
        output_path=scratch_output,
    )
    baseline['Reconstruction Log-Likelihood'] = _safe_metric_compute(
        ReconstructionLogLikelihood(),
        real_data=real_eval,
        reconstructed_data=recon_baseline,
        output_path=scratch_output,
    )
    baseline['Flexibility Distance'] = _safe_metric_compute(
        FlexibilityDistance(tau='auto', normalize=True),
        real_data=real_eval,
        reconstructed_data=recon_baseline,
        output_path=scratch_output,
    )

    baseline['Maximum Mean Discrepancy (MMD)'] = _safe_metric_compute(
        MaximumMeanDiscrepancy(sigma=None, seed=42, max_samples=2000),
        real_data=real_eval,
        synthetic_data=synth_baseline,
        output_path=scratch_output,
    )
    if variant == '3phase_15min':
        baseline['Set-MMD (Permutation Invariant)'] = _safe_metric_compute(
            SetMaximumMeanDiscrepancy(sigma=None, seed=42, max_samples=1000, n_features=3),
            real_data=real_eval,
            synthetic_data=synth_baseline,
            output_path=scratch_output,
        )

    baseline['ACF Distribution Match (KS)'] = _safe_metric_compute(
        ACFDistribution(window_size=window_size, max_lag=10),
        real_data=real_cons,
        synthetic_data=synth_cons,
        output_path=scratch_output,
    )
    baseline['Classical Statistics'] = _safe_metric_compute(
        ClassicalStatistics(window_size=window_size, n_features=1),
        real_data=real_cons,
        synthetic_data=synth_cons,
        output_path=scratch_output,
    )
    if not _SKIP_BASELINE_TSTR:
        baseline['TSTR (Train on Synthetic, Test on Real)'] = _safe_metric_compute(
            TSTR(task='next_step', window_size=window_size, n_features=1, predictor_type='linear', epochs=50, random_state=42),
            real_data=real_cons,
            synthetic_data=synth_cons,
            output_path=scratch_output,
        )
    baseline['Aggregated Profiles (Flexibility Distance)'] = _safe_metric_compute(
        AggregatedProfiles(n_samples=len(real_eval), window_size=window_size, tau='auto'),
        real_data=real_eval,
        synthetic_data=synth_baseline,
        output_path=scratch_output,
    )

    # Per-condition rows shown in this notebook (mmd/tstr) from full eval pool.
    required_labels = _required_per_condition_labels_all_models()
    per_cond = _compute_per_condition_baseline(real_cons_full, synth_cons_full, ts_full, required_labels=required_labels)
    baseline.update(per_cond)

    # Keep only successful numeric baseline values.
    baseline = {k: float(v) for k, v in baseline.items() if isinstance(v, (int, float, np.floating, np.integer)) and np.isfinite(float(v))}

    source = (
        f"computed-live:train-mean-as-generated-all-metrics(variant={variant}, "
        f"n_generated={n_generated}, source={n_generated_source}, n_eval_used={len(real_eval)})"
    )
    if _SKIP_BASELINE_TSTR:
        source += '+skip-baseline-tstr'
    _MEAN_BASELINE_CACHE[cache_key] = (baseline, source)
    return baseline, source

def prepare_shared_mean_baseline(shared_plan: dict, force_recompute: bool = False) -> tuple[dict, str]:
    key = _shared_baseline_key(shared_plan)
    if (not force_recompute) and key in _SHARED_MEAN_BASELINE_CACHE:
        return _SHARED_MEAN_BASELINE_CACHE[key]
    baseline, source = _compute_mean_generated_baseline_metrics(shared_plan, model_dir=None)
    source = f'{source}+shared-once'
    _SHARED_MEAN_BASELINE_CACHE[key] = (baseline, source)
    return baseline, source

def _get_or_create_baseline_metric_map(model_dir: Path, shared_plan: dict | None) -> tuple[dict, str]:
    if shared_plan is None:
        return {}, 'missing(no-shared-plan)'
    try:
        return prepare_shared_mean_baseline(shared_plan, force_recompute=False)
    except Exception as e:
        return {}, f'missing(mean-generated-failed:{e})'

print('Baseline override loaded: all metrics now use mean(training) as generated baseline with dynamic per-model synthetic count.')

Baseline override loaded: all metrics now use mean(training) as generated baseline with dynamic per-model synthetic count.


In [61]:
# Build one strict shared evaluation data plan so all models are evaluated on identical parquet files.
# Options: '1phase_hourly' or '3phase_15min'
DATA_VARIANT = '1phase_hourly'

shared_eval_plan = _build_shared_eval_plan(
    data_variant=DATA_VARIANT,
    max_files=None,
    seed=42,
    config_path=CONFIG_PATH,
    )

print('Shared evaluation data plan loaded.')
print(f"Config: {shared_eval_plan['config_path']}")
print(f"Data variant: {shared_eval_plan['data_variant']}")
print(f"Data root: {shared_eval_plan['data_root']}")
print(f"Parquet files: {len(shared_eval_plan['parquet_paths'])}")
print(f"Batch size: {shared_eval_plan['batch_size']}")
print(f"Eval batches: {shared_eval_plan['eval_batches']}")
print(f"Min buffer size: {shared_eval_plan['min_buffer_size']}")
print(f"Replay capacity: {shared_eval_plan['replay_buffer_capacity']}")

Shared evaluation data plan loaded.
Config: /Users/clarareginekold/Desktop/dtu/THESIS/rasmus-clara-synthetic-data/config/config.yaml
Data variant: 1phase_hourly
Data root: /Users/clarareginekold/Desktop/dtu/THESIS/rasmus-clara-synthetic-data/data/parq_hourly
Parquet files: 12
Batch size: 32
Eval batches: 5
Min buffer size: 256
Replay capacity: 2000


In [62]:
# Data load/preview from the selected shared eval parquet set
if not shared_eval_plan['parquet_paths']:
    raise RuntimeError('No parquet paths in shared_eval_plan. Re-run the shared_eval_plan cell.')
first_parquet = Path(shared_eval_plan['parquet_paths'][0])
preview_table = pq.read_table(first_parquet)
preview_df = preview_table.slice(0, 5).to_pandas()
print(f"Selected variant: {shared_eval_plan['data_variant']}")
print(f'First parquet file: {first_parquet}')
print(f'Rows in file: {preview_table.num_rows}, columns: {preview_table.num_columns}')
display(preview_df.head())

Selected variant: 1phase_hourly
First parquet file: /Users/clarareginekold/Desktop/dtu/THESIS/rasmus-clara-synthetic-data/data/parq_hourly/hourly_measurements_2023_10.parquet
Rows in file: 15937696, columns: 3


hourly_energy_consumption
meter_number timestamp_dst                                       
1000081      2023-10-01 00:00:00+00:00                       0.03
             2023-10-01 01:00:00+00:00                       0.04
             2023-10-01 02:00:00+00:00                       0.04
             2023-10-01 03:00:00+00:00                       0.03
             2023-10-01 04:00:00+00:00                       0.05

In [46]:
# Build shared mean baseline once (reuse for all model comparison cells).
shared_mean_baseline_metrics, shared_mean_baseline_source = prepare_shared_mean_baseline(shared_eval_plan, force_recompute=False)
print('Baseline source:', shared_mean_baseline_source)
display('Baseline metrics:', shared_mean_baseline_metrics)

Baseline source: computed-live:train-mean-as-generated-all-metrics(variant=1phase_hourly, n_generated=160, source=default(no-model-dir), n_eval_used=160)+shared-once


'Baseline metrics:'

{'Reconstruction MSE': 0.44801119015976515,
 'Reconstruction RMSE': 0.6693363804244956,
 'Reconstruction Log-Likelihood': -1.0174702513615315,
 'Flexibility Distance': 0.4727381883214351,
 'Maximum Mean Discrepancy (MMD)': 0.43747800372643275,
 'ACF Distribution Match (KS)': 0.680625,
 'Classical Statistics': 0.04543749952766413,
 'TSTR (Train on Synthetic, Test on Real)': 1.0687410668003758,
 'Aggregated Profiles (Flexibility Distance)': 7.16236099305875,
 'Per-Condition/mmd/day_of_week=1': 0.4946525627327976,
 'Per-Condition/mmd/month_of_year=1': 0.46258938819372375,
 'Per-Condition/tstr/day_of_week=1': 0.535598479661819,
 'Per-Condition/tstr/month_of_year=1': 2.721351540596119}

### Per-Model Evaluation Cells

Run the cells below one by one. Each cell tries to run eval+visualization via `load_checkpoint` and falls back to existing artifacts if the model type is unsupported there.

In [ ]:
# 1) AutoRegressiveTransformerVAE
result_art = evaluate_model_run(
    model_name='AutoRegressiveTransformerVAE_AutoregressiveTransformerCoder_20260415_163139',
    shared_plan=shared_eval_plan,
    rerun_if_supported=False,
    )


=== AutoRegressiveTransformerVAE_AutoregressiveTransformerCoder_20260415_163139 ===
Hyperparameters used
+----------------------------------+-----------------------------------------------------------------------------------------------------------------------+
| Key                              | Value                                                                                                                 |
+----------------------------------+-----------------------------------------------------------------------------------------------------------------------+
| config                           | models/hpc_runs/AutoRegressiveTransformerVAE_AutoregressiveTransformerCoder_20260415_163139/training_config_used.yaml |
| model.type                       | art                                                                                                                   |
| model.latent_dim                 | 64                                                                      

In [48]:
# 2) CVAE Dictionary
result_cvae_dict = evaluate_model_run(
    model_name='CVAE_DictionaryCoder_20260416_082400',
    shared_plan=shared_eval_plan,
    rerun_if_supported=False,
    )


=== CVAE_DictionaryCoder_20260416_082400 ===
Hyperparameters used
+-----------------------------+--------------------------------------------------------------------------------+
| Key                         | Value                                                                          |
+-----------------------------+--------------------------------------------------------------------------------+
| config                      | models/hpc_runs/CVAE_DictionaryCoder_20260416_082400/training_config_used.yaml |
| model.type                  | cvae                                                                           |
| model.latent_dim            | 64                                                                             |
| model.encoder.type          | dictionary                                                                     |
| model.decoder.type          | dictionary                                                                     |
| model.prior.type           

In [49]:
# 3) CVAE Dictionary + Hierarchy
result_cvae_hierarchy = evaluate_model_run(
    model_name='CVAE_DictionaryCoder_20260416_085408_hierarchy',
    shared_plan=shared_eval_plan,
    rerun_if_supported=False,
    )


=== CVAE_DictionaryCoder_20260416_085408_hierarchy ===
Hyperparameters used
+-----------------------------+------------------------------------------------------------------------------------------+
| Key                         | Value                                                                                    |
+-----------------------------+------------------------------------------------------------------------------------------+
| config                      | models/hpc_runs/CVAE_DictionaryCoder_20260416_085408_hierarchy/training_config_used.yaml |
| model.type                  | cvae                                                                                     |
| model.latent_dim            | 64                                                                                       |
| model.encoder.type          | dictionary                                                                               |
| model.decoder.type          | dictionary                    

In [63]:
# 4) CVAE Dictionary + Vamp Prior
# ras is running this again, as it didnt finish 
result_cvae_vamp = evaluate_model_run(
    model_name='CVAE_DictionaryCoder_20260416_230553_vamp',
    shared_plan=shared_eval_plan,
    rerun_if_supported=False,
    )


=== CVAE_DictionaryCoder_20260416_230553_vamp ===
Hyperparameters used
+-----------------------------+-------------------------------------------------------------------------------------+
| Key                         | Value                                                                               |
+-----------------------------+-------------------------------------------------------------------------------------+
| config                      | models/hpc_runs/CVAE_DictionaryCoder_20260416_230553_vamp/training_config_used.yaml |
| model.type                  | cvae                                                                                |
| model.latent_dim            | 64                                                                                  |
| model.encoder.type          | dictionary                                                                          |
| model.decoder.type          | dictionary                                                            

In [64]:
# 5) CVAE Gaussian
result_cvae_gauss = evaluate_model_run(
    model_name='CVAE_GaussianCoder_20260416_085358',
    shared_plan=shared_eval_plan,
    rerun_if_supported=False,
    )


=== CVAE_GaussianCoder_20260416_085358 ===
Hyperparameters used
+-----------------------------+------------------------------------------------------------------------------+
| Key                         | Value                                                                        |
+-----------------------------+------------------------------------------------------------------------------+
| config                      | models/hpc_runs/CVAE_GaussianCoder_20260416_085358/training_config_used.yaml |
| model.type                  | cvae                                                                         |
| model.latent_dim            | 64                                                                           |
| model.encoder.type          | mlp                                                                          |
| model.decoder.type          | mlp                                                                          |
| model.prior.type            | standard_gaussi

In [65]:
# 6) Diffusion
result_diffusion = evaluate_model_run(
    model_name='DiffusionModel_ConditionalUNet1D_20260415_172958',
    shared_plan=shared_eval_plan,
    # Diffusion is not supported by load_checkpoint phased rerun here; use saved artifacts.
    rerun_if_supported=False,
    )


=== DiffusionModel_ConditionalUNet1D_20260415_172958 ===
Hyperparameters used
+-------------------------------+--------------------------------------------------------------------------------------------+
| Key                           | Value                                                                                      |
+-------------------------------+--------------------------------------------------------------------------------------------+
| config                        | models/hpc_runs/DiffusionModel_ConditionalUNet1D_20260415_172958/training_config_used.yaml |
| model.type                    | diffusion                                                                                  |
| model.latent_dim              | 64                                                                                         |
| model.encoder.type            | mlp                                                                                        |
| model.decoder.type            

In [73]:
# 7) SetGuideVAE (3-phase) - use aggregated phase eval artifacts
setguide_model_dir = HPC_RUNS_DIR / 'SetGuideVAE_DictionaryCoder_20260416_103152'
setguide_config = setguide_model_dir / 'training_config_used.yaml'
shared_eval_plan_setguide = _build_shared_eval_plan(
    data_variant='3phase_15min',
    max_files=None,
    seed=42,
    config_path=setguide_config if setguide_config.exists() else CONFIG_PATH,
    )

print('\n=== SetGuideVAE_DictionaryCoder_20260416_103152 ===')
_print_model_hyperparameters(setguide_model_dir)

setguide_agg_eval_json = setguide_model_dir / 'aggregated_phase_eval' / 'eval_results.json'
if setguide_agg_eval_json.exists():
    print('Using aggregated SetGuide evaluation results.')
    result_setguide = _display_eval_results(setguide_agg_eval_json, shared_plan=shared_eval_plan_setguide)
    _show_visual_artifacts(setguide_agg_eval_json.parent)
else:
    print('Aggregated eval not found, falling back to top-level SetGuide eval_results.json.')
    result_setguide = evaluate_model_run(
        model_name='SetGuideVAE_DictionaryCoder_20260416_103152',
        shared_plan=shared_eval_plan_setguide,
        rerun_if_supported=False,
        )


=== SetGuideVAE_DictionaryCoder_20260416_103152 ===
Hyperparameters used
+-----------------------------+---------------------------------------------------------------------------------------+
| Key                         | Value                                                                                 |
+-----------------------------+---------------------------------------------------------------------------------------+
| config                      | models/hpc_runs/SetGuideVAE_DictionaryCoder_20260416_103152/training_config_used.yaml |
| model.type                  | set_guide_vae                                                                         |
| model.latent_dim            | 128                                                                                   |
| model.encoder.type          | dictionary_cnn                                                                        |
| model.decoder.type          | dictionary_cnn                                        

### Overall Comparison Across All HPC Runs

This cell collects each model folder's `eval_results.json` and compares them side-by-side using `compare_eval`.

In [81]:
import importlib
import pandas as pd
import evaluation.compare_eval as _compare_eval
importlib.reload(_compare_eval)
ModelComparator = _compare_eval.ModelComparator

comparison_output_dir = PROJECT_ROOT / 'evaluation_comparison_hpc_runs'
comparator = ModelComparator(output_dir=str(comparison_output_dir))

# Legend/display names used in comparison table + plots.
DISPLAY_MODEL_NAMES = {
    'AutoRegressiveTransformerVAE_AutoregressiveTransformerCoder_20260415_163139': 'ART-VAE',
    'CVAE_DictionaryCoder_20260416_082400': 'CVAE-Dict',
    'CVAE_DictionaryCoder_20260416_085408_hierarchy': 'CVAE-Hierarchy',
    'CVAE_DictionaryCoder_20260416_230553_vamp': 'CVAE-Vamp',
    'CVAE_GaussianCoder_20260416_085358': 'CVAE-Gaussian',
    'DiffusionModel_ConditionalUNet1D_20260415_172958': 'Diffusion',
    # 'SetGuideVAE_DictionaryCoder_20260416_103152': 'SetGuideVAE',
}

# Use aggregated SetGuide eval artifact for fair 3-phase comparison.
eval_paths = {}
for model_name in MODEL_RUNS:
    model_dir = HPC_RUNS_DIR / model_name
    if model_name.startswith('SetGuideVAE_'):
        eval_json = model_dir / 'aggregated_phase_eval' / 'eval_results.json'
    else:
        eval_json = model_dir / 'eval_results.json'
    eval_paths[model_name] = eval_json

added = []
for model_name in MODEL_RUNS:
    eval_json = eval_paths[model_name]
    if eval_json.exists():
        display_name = DISPLAY_MODEL_NAMES.get(model_name, model_name)
        comparator.add_model(display_name, str(eval_json))
        added.append(display_name)
    else:
        print(f'Skipping {model_name}: eval results not found at {eval_json}')

if not added:
    print('No eval result files found. Run model cells first.')
else:
    print(f'Comparing {len(added)} models with ModelComparator...')
    print('Legend names:', ', '.join(added))

    # Readable table in notebook output (independent of tabulate availability).
    all_metrics = sorted({m for mm in comparator.models_metrics.values() for m in mm.keys()})
    table_df = pd.DataFrame(index=all_metrics, columns=added, dtype=float)
    for model_name in added:
        for metric_name, metric_val in comparator.models_metrics.get(model_name, {}).items():
            table_df.loc[metric_name, model_name] = metric_val
    table_df.index.name = 'Metric'

    print('\nReadable model comparison table:')
    display(table_df.round(2))

    comparator.compare()
    print(f'Comparison artifacts written to: {comparison_output_dir}')

Comparing 7 models with ModelComparator...
Legend names: ART-VAE, CVAE-Dict, CVAE-Hierarchy, CVAE-Vamp, CVAE-Gaussian, Diffusion, SetGuideVAE_DictionaryCoder_20260416_103152

Readable model comparison table:


,ART-VAE,CVAE-Dict,CVAE-Hierarchy,CVAE-Vamp,CVAE-Gaussian,Diffusion,SetGuideVAE_DictionaryCoder_20260416_103152
Metric,,,,,,,
ACF Distribution Match (KS),0.13,0.10,0.10,0.16,0.17,0.07,0.09
Aggregated Profiles (Flexibility Distance),113.98,157.28,132.22,143.87,115.70,999825.37,130.28
Classical Statistics,0.05,1.94,0.08,0.29,0.04,33205.70,1.41
Flexibility Distance,0.52,0.51,0.58,0.54,0.59,0.29,0.82
Maximum Mean Discrepancy (MMD),0.07,0.12,0.12,0.21,0.28,0.15,0.10
Per-Condition Evaluation,0.59,0.14,0.20,0.30,0.32,0.18,0.42
Reconstruction Log-Likelihood,-1.59,-1.61,-2.05,-1.62,-4.84,-1.08,-1.80
Reconstruction RMSE,1.19,1.21,1.88,1.22,30.48,0.71,1.47
"TSTR (Train on Synthetic, Test on Real)",1.26,0.50,0.58,1.20,1.44,0.19,1.57



Model Comparison Table:
[['ACF Distribution Match (KS) (↓)', '0.1298', '0.1040', '0.1014', '0.1553', '0.1720', '0.0679', '0.0920'], ['Aggregated Profiles (Flexibility Distance) (↓)', '113.9843', '157.2807', '132.2205', '143.8745', '115.6963', '999825.3675', '130.2774'], ['Classical Statistics (↓)', '0.0549', '1.9444', '0.0781', '0.2927', '0.0398', '33205.7026', '1.4137'], ['Flexibility Distance (↓)', '0.5184', '0.5089', '0.5827', '0.5376', '0.5939', '0.2905', '0.8208'], ['Maximum Mean Discrepancy (MMD) (0)', '0.0717', '0.1191', '0.1191', '0.2096', '0.2812', '0.1471', '0.1028'], ['Per-Condition Evaluation (↓)', '0.5894', '0.1439', '0.1983', '0.2982', '0.3216', '0.1826', '0.4236'], ['Reconstruction Log-Likelihood (↑)', '-1.5896', '-1.6059', '-2.0503', '-1.6201', '-4.8359', '-1.0823', '-1.8040'], ['Reconstruction RMSE (↓)', '1.1861', '1.2053', '1.8801', '1.2227', '30.4775', '0.7095', '1.4698'], ['TSTR (Train on Synthetic, Test on Real) (1)', '1.2592', '0.5048', '0.5791', '1.1978', '1.435